java -jar PerformanceMatcher.jar -b -q -D --use-chroma-map -os alignment_path.txt /Users/server/Desktop/match/wav/bach5_mov1_S_tsm.wav /Users/server/Desktop/match/wav/bach5_mov1_S.wav >> dump_features_testagain.txt

In [ ]:
import subprocess
import tempfile
import os
import pickle
import numpy as np
import re

In [9]:
def run_match(output_path="match/offline_alignment_path.txt", wav1_path=None, wav2_path=None, timeout=1200):
    if wav1_path is None:
        print("Please provide a query wav file path")
        return None
    if wav2_path is None:
        print("Please provide a reference wav file path")
        return None
    
    # Check Java version - PerformanceMatcher requires Java 21+
    java_executable = "java"
    java_found = False
    
    # First, try to find Java 21+ in common locations
    java_candidates = [
        "/usr/lib/jvm/java-21-openjdk-amd64/bin/java",
        "/usr/lib/jvm/java-21-openjdk/bin/java",
        "java"  # Fallback to default
    ]
    
    for java_candidate in java_candidates:
        try:
            java_version_output = subprocess.run(
                [java_candidate, "-version"],
                capture_output=True,
                text=True,
                timeout=5
            )
            # Java version outputs to stderr, not stdout
            version_str = java_version_output.stderr or java_version_output.stdout
            version_match = re.search(r'version ["\']?(\d+)', version_str)
            if version_match:
                java_version = int(version_match.group(1))
                if java_version >= 21:
                    java_executable = java_candidate
                    java_found = True
                    print(f"Using Java {java_version} from {java_executable}")
                    break
        except (FileNotFoundError, subprocess.TimeoutExpired) as e:
            continue
        except Exception as e:
            continue
    
    if not java_found:
        # Check default Java and warn if it's too old
        try:
            java_version_output = subprocess.run(
                ["java", "-version"],
                capture_output=True,
                text=True,
                timeout=5
            )
            version_str = java_version_output.stderr or java_version_output.stdout
            version_match = re.search(r'version ["\']?(\d+)', version_str)
            if version_match:
                java_version = int(version_match.group(1))
                if java_version < 21:
                    print(f"ERROR: PerformanceMatcher requires Java 21 or later, but found Java {java_version}")
                    print("Please install Java 21 with:")
                    print("  sudo apt-get update")
                    print("  sudo apt-get install -y openjdk-21-jdk")
                    print("Then either:")
                    print("  sudo update-alternatives --config java  # to set Java 21 as default")
                    print("  OR the function will automatically use Java 21 if installed")
                    return None
        except Exception as e:
            print(f"Warning: Could not check Java version: {e}")
    
    # Get the absolute path to the project root
    # Try to find project root by looking for match/PerformanceMatcher.jar
    current_dir = os.getcwd()
    # If we're in match/ directory, go up one level
    if os.path.basename(current_dir) == "match":
        project_root = os.path.dirname(current_dir)
    else:
        project_root = current_dir
    
    # Verify project root by checking for the jar file
    jar_path = os.path.join(project_root, "match", "PerformanceMatcher.jar")
    if not os.path.exists(jar_path):
        # Fallback: try current directory structure
        jar_path = os.path.abspath("match/PerformanceMatcher.jar")
        if os.path.exists(jar_path):
            project_root = os.path.dirname(os.path.dirname(jar_path))
            jar_path = os.path.join(project_root, "match", "PerformanceMatcher.jar")
        else:
            print(f"ERROR: Could not find PerformanceMatcher.jar")
            print(f"  Tried: {os.path.join(project_root, 'match', 'PerformanceMatcher.jar')}")
            print(f"  Tried: {jar_path}")
            return None
    
    # Convert paths to absolute
    wav1_path = os.path.abspath(wav1_path)
    wav2_path = os.path.abspath(wav2_path)
    if not os.path.exists(wav1_path):
        print(f"ERROR: Could not find wav file: {wav1_path}")
        return None
    if not os.path.exists(wav2_path):
        print(f"ERROR: Could not find wav file: {wav2_path}")
        return None
    
    if output_path is None:
        tmp_file = tempfile.NamedTemporaryFile(delete=False, suffix=".txt")
        output_path = tmp_file.name
        tmp_file.close()
    else:
        tmp_file = None
        output_path = os.path.abspath(output_path)
    
    cmd = [
        java_executable,
        "-jar",
        jar_path,
        "-b",
        "-q",
        "-D",
        "--use-chroma-map",
        "-os",
        output_path,
        wav1_path,
        wav2_path
    ]
    
    # Ensure dump_alignment.txt directory exists
    dump_file = os.path.join(project_root, "match", "dump_alignment.txt")
    os.makedirs(os.path.dirname(dump_file), exist_ok=True)

    try:
        # Open dump file for appending and redirect stdout/stderr
        with open(dump_file, "a") as dump_f:
            result = subprocess.run(
                cmd,
                check=True,
                stdout=dump_f,
                stderr=subprocess.STDOUT,  # Redirect stderr to stdout so it's captured
                timeout=timeout,
                cwd=project_root  # Run from project root
            )
        return output_path
    except subprocess.TimeoutExpired:
        print(f"PerformanceMatcher timed out after {timeout} seconds.")
        if tmp_file:
            try:
                os.remove(output_path)
            except Exception as e:
                print(f"Could not delete temp file: {e}")
        return None
    except subprocess.CalledProcessError as e:
        print(f"PerformanceMatcher failed with error: {e}")
        # Try to read and display the error from the dump file
        if os.path.exists(dump_file):
            with open(dump_file, "r") as f:
                lines = f.readlines()
                if lines:
                    print("Last few lines from dump_alignment.txt:")
                    for line in lines[-10:]:
                        print(f"  {line.rstrip()}")
        if tmp_file:
            try:
                os.remove(output_path)
            except Exception as ex:
                print(f"Could not delete temp file: {ex}")
        return None

In [10]:
run_match(wav1_path="scenarios/random/s1/p.wav",wav2_path="scenarios/random/s1/pref.wav")

ERROR: Could not find wav file: /mnt/data0/slubis/PianoConcertoAccompaniment/match/scenarios/random/s1/p.wav
